In [1]:
print(2+2)

4


In [2]:
"""
seed_rental_db.py
=================
Generates realistic synthetic data for the Rental Analytics project.
Outputs:
  - rental_analytics.db  (SQLite — works with DB Browser, pandas, Tableau)
  - /csv/                 (one CSV per table — easy Tableau import)

Designed to tell a real story:
  - ~2,000 products across 8 categories with FNAC-style pricing
  - Realistic inventory aging (log-normal sell-through times)
  - Rental demand varies by category and season
  - Customer churn risk correlates with late-return history
  - Revenue comparison data supports the core hypothesis

Run:
  python seed_rental_db.py
"""

import sqlite3
import os
import random
import math
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
from faker import Faker
fake = Faker("pt_PT")

random.seed(42)
np.random.seed(42)


In [3]:
# ── Output paths ──────────────────────────────────────────────────────────────
DB_PATH  = Path("rental_analytics.db")
CSV_DIR  = Path("csv")
CSV_DIR.mkdir(exist_ok=True)

# ── Simulation window ─────────────────────────────────────────────────────────
SIM_START = date(2022, 1, 1)
SIM_END   = date(2024, 12, 31)
TODAY     = date(2025, 4, 6)

def rand_date(start: date, end: date) -> date:
    delta = (end - start).days
    return start + timedelta(days=random.randint(0, delta))

def days_between(d1: date, d2: date) -> int:
    return (d2 - d1).days

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
#  CATEGORY DEFINITIONS
#  Each entry drives product generation + pricing rules + rental demand
# ─────────────────────────────────────────────────────────────────────────────
CATEGORIES = [
    {
        "name": "Televisions",
        "segment": "consumer_electronics",
        "avg_days_to_sale": 210,      # slow movers
        "sale_days_sigma": 0.9,       # log-normal shape
        "n_products": 280,
        "price_range": (299, 2499),
        "brands": ["Samsung", "LG", "Sony", "Philips", "TCL", "Hisense"],
        "rental_demand_mu": 3.2,      # avg rentals/month at peak
        "rental_duration_mu": 14,     # avg days per rental
        "rental_duration_sigma": 5,
    },
    {
        "name": "Drones",
        "segment": "consumer_electronics",
        "avg_days_to_sale": 280,
        "sale_days_sigma": 1.0,
        "n_products": 180,
        "price_range": (149, 1899),
        "brands": ["DJI", "Parrot", "Autel", "Holy Stone", "Ruko"],
        "rental_demand_mu": 4.5,
        "rental_duration_mu": 5,
        "rental_duration_sigma": 2,
    },
    {
        "name": "Gaming Consoles",
        "segment": "gaming",
        "avg_days_to_sale": 120,
        "sale_days_sigma": 0.7,
        "n_products": 200,
        "price_range": (249, 699),
        "brands": ["Sony", "Microsoft", "Nintendo"],
        "rental_demand_mu": 6.0,
        "rental_duration_mu": 10,
        "rental_duration_sigma": 4,
    },
    {
        "name": "Laptops",
        "segment": "computing",
        "avg_days_to_sale": 190,
        "sale_days_sigma": 0.85,
        "n_products": 320,
        "price_range": (399, 2999),
        "brands": ["Apple", "Dell", "HP", "Lenovo", "Asus", "Acer", "MSI"],
        "rental_demand_mu": 5.5,
        "rental_duration_mu": 21,
        "rental_duration_sigma": 7,
    },
    {
        "name": "Cameras & Photography",
        "segment": "consumer_electronics",
        "avg_days_to_sale": 240,
        "sale_days_sigma": 0.95,
        "n_products": 220,
        "price_range": (199, 3499),
        "brands": ["Canon", "Nikon", "Sony", "Fujifilm", "Panasonic", "Olympus"],
        "rental_demand_mu": 5.0,
        "rental_duration_mu": 7,
        "rental_duration_sigma": 3,
    },
    {
        "name": "Smart Home & Audio",
        "segment": "consumer_electronics",
        "avg_days_to_sale": 160,
        "sale_days_sigma": 0.75,
        "n_products": 260,
        "price_range": (49, 899),
        "brands": ["Sonos", "Bose", "Amazon", "Google", "Apple", "JBL", "Bang & Olufsen"],
        "rental_demand_mu": 2.8,
        "rental_duration_mu": 30,
        "rental_duration_sigma": 10,
    },
    {
        "name": "Keyboards & Peripherals",
        "segment": "computing",
        "avg_days_to_sale": 130,
        "sale_days_sigma": 0.65,
        "n_products": 240,
        "price_range": (29, 399),
        "brands": ["Logitech", "Razer", "Corsair", "SteelSeries", "Keychron", "Das Keyboard"],
        "rental_demand_mu": 2.0,
        "rental_duration_mu": 14,
        "rental_duration_sigma": 5,
    },
    {
        "name": "Projectors",
        "segment": "consumer_electronics",
        "avg_days_to_sale": 300,
        "sale_days_sigma": 1.05,
        "n_products": 180,
        "price_range": (199, 4999),
        "brands": ["Epson", "Optoma", "BenQ", "LG", "Sony", "Anker"],
        "rental_demand_mu": 6.5,     # high rental demand — events, presentations
        "rental_duration_mu": 3,
        "rental_duration_sigma": 1,
    },
]

In [5]:
# Adjectives for product name generation
ADJECTIVES = [
    "Ultra", "Pro", "Max", "Plus", "Elite", "Smart", "Advanced",
    "Premium", "Slim", "Compact", "Wireless", "4K", "HD", "Mini",
    "Neo", "Air", "Edge", "Flex", "Vision",
]
TV_SIZES    = [43, 50, 55, 65, 75, 85]
TV_TYPES    = ["QLED", "OLED", "LED", "Neo QLED", "MiniLED", "AMOLED"]
DRONE_TYPES = ["Nano", "Mini 3", "Mini 4", "Air 3", "Pro", "Enterprise"]
CONSOLE_GEN = {
    "Sony": ["PlayStation 5", "PlayStation 5 Digital", "PlayStation 4 Pro"],
    "Microsoft": ["Xbox Series X", "Xbox Series S", "Xbox One X"],
    "Nintendo": ["Switch OLED", "Switch", "Switch Lite"],
}

def make_product_name(cat_name: str, brand: str) -> str:
    adj  = random.choice(ADJECTIVES)
    year = random.choice([2021, 2022, 2023, 2024])
    model_num = f"{random.randint(1,9)}{''.join([str(random.randint(0,9)) for _ in range(2)])}"

    if cat_name == "Televisions":
        sz = random.choice(TV_SIZES)
        tp = random.choice(TV_TYPES)
        return f"{brand} {sz}\" {tp} {adj} {year}"
    elif cat_name == "Drones":
        tp = random.choice(DRONE_TYPES)
        return f"{brand} {tp} {adj}"
    elif cat_name == "Gaming Consoles":
        options = CONSOLE_GEN.get(brand, [f"{brand} Console"])
        return random.choice(options)
    elif cat_name == "Laptops":
        return f"{brand} {adj} {model_num} ({year})"
    elif cat_name == "Cameras & Photography":
        return f"{brand} {adj} {model_num} {'Mirrorless' if random.random() > 0.4 else 'DSLR'}"
    elif cat_name == "Smart Home & Audio":
        tp = random.choice(["Speaker", "Soundbar", "Smart Display", "Smart Hub", "Subwoofer"])
        return f"{brand} {adj} {tp}"
    elif cat_name == "Keyboards & Peripherals":
        tp = random.choice(["Mechanical Keyboard", "Gaming Mouse", "Wireless Combo",
                            "USB-C Hub", "Webcam", "Headset"])
        return f"{brand} {adj} {tp}"
    elif cat_name == "Projectors":
        return f"{brand} {adj} {model_num} {'4K' if random.random() > 0.5 else 'FHD'} Projector"
    return f"{brand} {adj} {model_num}"

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
#  BUILD TABLES
# ─────────────────────────────────────────────────────────────────────────────

def build_categories() -> pd.DataFrame:
    rows = []
    for i, c in enumerate(CATEGORIES, start=1):
        rows.append({
            "category_id":      i,
            "name":             c["name"],
            "segment":          c["segment"],
            "avg_days_to_sale": c["avg_days_to_sale"],
        })
    return pd.DataFrame(rows)


def build_products(cats_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    sku_counter = 10000
    conditions  = ["new"] * 80 + ["open_box"] * 15 + ["refurbished"] * 5  # weighted

    for cat_idx, cat in enumerate(CATEGORIES, start=1):
        for _ in range(cat["n_products"]):
            brand = random.choice(cat["brands"])
            name  = make_product_name(cat["name"], brand)

            lo, hi = cat["price_range"]
            retail_price = round(random.uniform(lo, hi) / 5) * 5  # rounded to nearest €5

            # Listed date spread across simulation window
            listed_date = rand_date(SIM_START, SIM_END - timedelta(days=30))

            # Time-to-sale drawn from log-normal
            mu_days = cat["avg_days_to_sale"]
            sigma   = cat["sale_days_sigma"]
            ln_mu   = math.log(mu_days) - (sigma**2) / 2
            days_to_sale = int(np.random.lognormal(ln_mu, sigma))
            days_to_sale = max(30, min(days_to_sale, 1200))

            sell_date     = listed_date + timedelta(days=days_to_sale)
            days_on_shelf = days_between(listed_date, TODAY)

            # Determine status
            if sell_date <= TODAY:
                # Sold — happened in the past
                status = "sold"
            elif days_on_shelf >= 365:
                # Still here, past the threshold
                status = random.choices(
                    ["eligible_for_rental", "rented", "eligible_for_rental"],
                    weights=[50, 35, 15]
                )[0]
            else:
                status = "for_sale"

            rows.append({
                "product_id":       len(rows) + 1,
                "category_id":      cat_idx,
                "sku":              f"SKU-{sku_counter}",
                "name":             name,
                "brand":            brand,
                "retail_price_eur": retail_price,
                "release_date":     (listed_date - timedelta(days=random.randint(0, 90))).isoformat(),
                "listed_date":      listed_date.isoformat(),
                "condition":        random.choice(conditions),
                "status":           status,
                "days_on_shelf":    days_on_shelf,
            })
            sku_counter += random.randint(1, 9)

    return pd.DataFrame(rows)

In [7]:
def build_inventory_events(products_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    event_id = 1

    for _, p in products_df.iterrows():
        listed  = date.fromisoformat(p["listed_date"])
        price   = float(p["retail_price_eur"])
        pid     = int(p["product_id"])
        status  = p["status"]

        # Always: listed event
        rows.append({
            "event_id":       event_id,
            "product_id":     pid,
            "event_type":     "listed",
            "event_date":     listed.isoformat(),
            "price_at_event": price,
            "notes":          "Initial stock entry",
        })
        event_id += 1

        days_on_shelf = int(p["days_on_shelf"])

        # Price reductions: happen ~40% of time after 6 months
        reduction_window = min(days_on_shelf - 30, 360)
        if days_on_shelf > 210 and reduction_window > 180 and random.random() < 0.4:
            reduction_date = listed + timedelta(days=random.randint(180, reduction_window))
            reduced_price  = round(price * random.uniform(0.70, 0.90), 2)
            rows.append({
                "event_id":       event_id,
                "product_id":     pid,
                "event_type":     "price_reduced",
                "event_date":     reduction_date.isoformat(),
                "price_at_event": reduced_price,
                "notes":          f"Seasonal markdown to {reduced_price}€",
            })
            event_id += 1

        # Rental eligible threshold crossed
        if days_on_shelf >= 365:
            eligible_date = listed + timedelta(days=365)
            rows.append({
                "event_id":       event_id,
                "product_id":     pid,
                "event_type":     "rental_eligible",
                "event_date":     eligible_date.isoformat(),
                "price_at_event": price,
                "notes":          "Crossed 12-month threshold — eligible for rental program",
            })
            event_id += 1

        if status == "rented":
            rows.append({
                "event_id":       event_id,
                "product_id":     pid,
                "event_type":     "rented_out",
                "event_date":     (listed + timedelta(days=days_on_shelf - random.randint(5, 60))).isoformat(),
                "price_at_event": price,
                "notes":          "Assigned to rental program",
            })
            event_id += 1

        if status == "sold":
            rows.append({
                "event_id":       event_id,
                "product_id":     pid,
                "event_type":     "sold",
                "event_date":     (listed + timedelta(days=days_on_shelf)).isoformat(),
                "price_at_event": round(price * random.uniform(0.55, 1.0), 2),
                "notes":          "Final sale",
            })
            event_id += 1

    return pd.DataFrame(rows)


In [8]:
def build_pricing_rules(cats_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    # Daily rate as % of retail varies by category risk/demand
    rate_map = {
        "Televisions":          0.0045,
        "Drones":               0.0060,
        "Gaming Consoles":      0.0055,
        "Laptops":              0.0040,
        "Cameras & Photography":0.0065,
        "Smart Home & Audio":   0.0035,
        "Keyboards & Peripherals": 0.0030,
        "Projectors":           0.0070,
    }
    deposit_map = {
        "Televisions":          0.25,
        "Drones":               0.30,
        "Gaming Consoles":      0.20,
        "Laptops":              0.25,
        "Cameras & Photography":0.30,
        "Smart Home & Audio":   0.15,
        "Keyboards & Peripherals": 0.15,
        "Projectors":           0.25,
    }
    for _, cat in cats_df.iterrows():
        cname = cat["name"]
        rows.append({
            "rule_id":              int(cat["category_id"]),
            "category_id":          int(cat["category_id"]),
            "age_min_days":         365,
            "age_max_days":         None,
            "base_daily_rate_pct":  rate_map.get(cname, 0.005),
            "deposit_pct":          deposit_map.get(cname, 0.20),
            "late_fee_daily":       random.choice([3.0, 5.0, 7.5, 10.0]),
            "active":               1,
        })
    return pd.DataFrame(rows)


In [9]:
def build_customers(n: int = 800) -> pd.DataFrame:
    rows = []
    for i in range(1, n + 1):
        total_rentals = random.randint(0, 18)
        late_returns  = int(np.random.binomial(total_rentals, 0.15)) if total_rentals > 0 else 0
        avg_return_days = round(np.random.normal(12, 4), 1) if total_rentals > 0 else None

        # Churn risk: higher if more late returns relative to total
        late_ratio = late_returns / max(total_rentals, 1)
        churn_base = late_ratio * 0.6 + random.uniform(0, 0.4)
        churn_risk = round(min(max(churn_base, 0.0), 1.0), 3)

        nif_digits = "".join([str(random.randint(0, 9)) for _ in range(9)])

        rows.append({
            "customer_id":      i,
            "full_name":        fake.name(),
            "email":            fake.email(),
            "nif":              nif_digits,
            "phone":            fake.phone_number(),
            "address":          fake.address().replace("\n", ", "),
            "id_verified":      1 if random.random() > 0.1 else 0,
            "total_rentals":    total_rentals,
            "late_returns":     late_returns,
            "avg_return_days":  avg_return_days,
            "churn_risk_score": churn_risk,
        })
    return pd.DataFrame(rows)

In [10]:
def build_rentals(products_df: pd.DataFrame,
                  customers_df: pd.DataFrame,
                  pricing_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:

    # Only products that were eligible / rented / have long shelf time
    rentable = products_df[
        products_df["status"].isin(["eligible_for_rental", "rented"]) |
        (products_df["days_on_shelf"] >= 365)
    ].copy()

    # Build a lookup: category_id → pricing rule
    price_lookup = {
        int(r["category_id"]): r
        for _, r in pricing_df.iterrows()
    }

    rental_rows    = []
    condition_rows = []
    rental_id      = 1
    condition_id   = 1

    cat_demand = {c["name"]: c["rental_demand_mu"] for c in CATEGORIES}
    cat_dur    = {c["name"]: (c["rental_duration_mu"], c["rental_duration_sigma"]) for c in CATEGORIES}
    cat_name_map = {i+1: c["name"] for i, c in enumerate(CATEGORIES)}

In [11]:
def build_rentals(products_df: pd.DataFrame,
                  customers_df: pd.DataFrame,
                  pricing_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:

    # Only products that were eligible / rented / have long shelf time
    rentable = products_df[
        products_df["status"].isin(["eligible_for_rental", "rented"]) |
        (products_df["days_on_shelf"] >= 365)
    ].copy()

    # Build a lookup: category_id → pricing rule
    price_lookup = {
        int(r["category_id"]): r
        for _, r in pricing_df.iterrows()
    }

    rental_rows    = []
    condition_rows = []
    rental_id      = 1
    condition_id   = 1

    cat_demand = {c["name"]: c["rental_demand_mu"] for c in CATEGORIES}
    cat_dur    = {c["name"]: (c["rental_duration_mu"], c["rental_duration_sigma"]) for c in CATEGORIES}
    cat_name_map = {i+1: c["name"] for i, c in enumerate(CATEGORIES)}

    # Generate rentals across the simulation window
    sim_months = pd.date_range(SIM_START.isoformat(), SIM_END.isoformat(), freq="MS")

    for month_start in sim_months:
        month_date = month_start.date()

        for _, product in rentable.sample(frac=0.15, random_state=random.randint(0, 9999)).iterrows():
            cat_id   = int(product["category_id"])
            cat_name = cat_name_map.get(cat_id, "Unknown")
            rule     = price_lookup.get(cat_id)
            if rule is None:
                continue

            retail = float(product["retail_price_eur"])
            daily_rate = round(retail * float(rule["base_daily_rate_pct"]), 2)
            deposit    = round(retail * float(rule["deposit_pct"]), 2)

            # How many times rented this month?
            demand_mu  = cat_demand.get(cat_name, 3.0)
            n_rentals  = max(0, int(np.random.poisson(demand_mu / 10)))

            for _ in range(n_rentals):
                if rental_id > 3500:
                    break

                dur_mu, dur_sigma = cat_dur.get(cat_name, (10, 4))
                duration = max(1, int(np.random.normal(dur_mu, dur_sigma)))

                rental_start = month_date + timedelta(days=random.randint(0, 27))
                rental_due   = rental_start + timedelta(days=duration)

                # Has it been returned?
                is_past = rental_due < TODAY
                if is_past:
                    # 85% returned; 8% late; 7% still out
                    outcome = random.choices(
                        ["on_time", "late", "still_out"],
                        weights=[75, 17, 8]
                    )[0]
                else:
                    outcome = "still_out"

                late_days = 0
                if outcome == "on_time":
                    rental_returned = rental_due - timedelta(days=random.randint(0, 2))
                    status = "returned"
                elif outcome == "late":
                    late_days = random.randint(1, 14)
                    rental_returned = rental_due + timedelta(days=late_days)
                    status = "returned"
                else:
                    rental_returned = None
                    status = "active" if rental_due >= TODAY else "overdue"

                days_held = (
                    days_between(rental_start, rental_returned)
                    if rental_returned else
                    days_between(rental_start, TODAY)
                )

                late_fee   = float(rule["late_fee_daily"]) * late_days
                total_charged = round(daily_rate * days_held + late_fee, 2)

                customer = customers_df.sample(1).iloc[0]

                rental_rows.append({
                    "rental_id":       rental_id,
                    "product_id":      int(product["product_id"]),
                    "customer_id":     int(customer["customer_id"]),
                    "rental_start":    rental_start.isoformat(),
                    "rental_due":      rental_due.isoformat(),
                    "rental_returned": rental_returned.isoformat() if rental_returned else None,
                    "daily_rate_eur":  daily_rate,
                    "deposit_eur":     deposit,
                    "days_held":       days_held,
                    "total_charged":   total_charged,
                    "status":          status,
                })

                # Return condition (only for returned rentals)
                if rental_returned is not None:
                    cond = random.choices(
                        ["pristine", "good", "fair", "damaged", "destroyed"],
                        weights=[30, 45, 18, 6, 1]
                    )[0]
                    damage = 0.0
                    dep_returned = True
                    if cond == "damaged":
                        damage = round(random.uniform(20, retail * 0.3), 2)
                        dep_returned = random.random() > 0.5
                    elif cond == "destroyed":
                        damage = round(retail * random.uniform(0.5, 1.0), 2)
                        dep_returned = False

                    condition_rows.append({
                        "condition_id":        condition_id,
                        "rental_id":           rental_id,
                        "assessed_condition":  cond,
                        "damage_charge":       damage,
                        "deposit_returned":    1 if dep_returned else 0,
                        "inspector_notes":     f"Assessed after return on {rental_returned.isoformat()}",
                    })
                    condition_id += 1

                rental_id += 1

    return pd.DataFrame(rental_rows), pd.DataFrame(condition_rows)


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
#  WRITE TO SQL + CSV
# ─────────────────────────────────────────────────────────────────────────────

def write_all():
    print("Building tables...")

    from sqlalchemy import create_engine, text
    from dotenv import load_dotenv
    import os

    load_dotenv()

    engine = create_engine(
        f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
        f"@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}"
    )

    cats_df      = build_categories()
    products_df  = build_products(cats_df)
    events_df    = build_inventory_events(products_df)
    pricing_df   = build_pricing_rules(cats_df)
    customers_df = build_customers(800)
    rentals_df, conditions_df = build_rentals(products_df, customers_df, pricing_df)

    tables = {
        "categories":        cats_df,
        "products":          products_df,
        "inventory_events":  events_df,
        "pricing_rules":     pricing_df,
        "customers":         customers_df,
        "rentals":           rentals_df,
        "return_conditions": conditions_df,
    }

    # drop tables safely first, ignoring foreign key constraints
    with engine.connect() as con:
        con.execute(text("SET FOREIGN_KEY_CHECKS = 0"))
        for table_name in tables.keys():
            con.execute(text(f"DROP TABLE IF EXISTS `{table_name}`"))
        con.execute(text("SET FOREIGN_KEY_CHECKS = 1"))
        con.commit()

    for table_name, df in tables.items():
        df.to_sql(table_name, engine, if_exists="replace", index=False)
        print(f"  {table_name:25s}  {len(df):>6,} rows")

    # ── Analytical views ───────────────────────────────────────
    views = [
        "DROP VIEW IF EXISTS v_rental_eligible",
        """CREATE VIEW v_rental_eligible AS
        SELECT
            p.product_id, p.sku, p.name, p.brand,
            c.name AS category, c.segment,
            p.retail_price_eur, p.listed_date,
            DATEDIFF(CURDATE(), p.listed_date) AS days_on_shelf,
            p.condition, pr.base_daily_rate_pct,
            ROUND(p.retail_price_eur * pr.base_daily_rate_pct, 2)      AS daily_rate_eur,
            ROUND(p.retail_price_eur * pr.deposit_pct, 2)              AS deposit_eur,
            ROUND(p.retail_price_eur * 0.60, 2)                        AS markdown_revenue,
            ROUND(p.retail_price_eur * pr.base_daily_rate_pct * 90, 2) AS rental_90d_revenue
        FROM products p
        JOIN categories c     ON p.category_id = c.category_id
        JOIN pricing_rules pr ON pr.category_id = c.category_id
        WHERE p.status IN ('eligible_for_rental', 'rented') AND pr.active = 1""",

        "DROP VIEW IF EXISTS v_rental_history",
        """CREATE VIEW v_rental_history AS
        SELECT
            r.rental_id,
            p.name AS product_name, c.name AS category, cu.full_name AS customer_name,
            r.rental_start, r.rental_due, r.rental_returned,
            DATEDIFF(COALESCE(r.rental_returned, CURDATE()), r.rental_start) AS days_held,
            r.daily_rate_eur, r.deposit_eur, r.total_charged, r.status,
            rc.assessed_condition, rc.damage_charge, rc.deposit_returned,
            CASE WHEN r.rental_returned > r.rental_due THEN 1 ELSE 0 END AS returned_late,
            DATEDIFF(COALESCE(r.rental_returned, CURDATE()), r.rental_due) AS days_overdue
        FROM rentals r
        JOIN products p   ON r.product_id  = p.product_id
        JOIN categories c ON p.category_id = c.category_id
        JOIN customers cu ON r.customer_id = cu.customer_id
        LEFT JOIN return_conditions rc ON r.rental_id = rc.rental_id""",

        "DROP VIEW IF EXISTS v_inventory_aging",
        """CREATE VIEW v_inventory_aging AS
        SELECT
            c.name AS category, c.segment,
            COUNT(*) AS total_items,
            ROUND(AVG(DATEDIFF(CURDATE(), p.listed_date)), 1) AS avg_days_on_shelf,
            SUM(p.retail_price_eur) AS total_retail_value_eur,
            SUM(CASE WHEN DATEDIFF(CURDATE(), p.listed_date) > 365
                     THEN p.retail_price_eur ELSE 0 END) AS stale_inventory_value_eur,
            COUNT(CASE WHEN p.status = 'eligible_for_rental' THEN 1 END) AS rental_eligible_count,
            COUNT(CASE WHEN p.status = 'sold' THEN 1 END) AS sold_count
        FROM products p
        JOIN categories c ON p.category_id = c.category_id
        GROUP BY c.name, c.segment""",

        "DROP VIEW IF EXISTS v_revenue_comparison",
        """CREATE VIEW v_revenue_comparison AS
        SELECT
            c.name AS category,
            COUNT(DISTINCT r.rental_id) AS total_rentals,
            ROUND(SUM(r.total_charged), 2) AS total_rental_revenue,
            ROUND(AVG(r.total_charged), 2) AS avg_revenue_per_rental,
            ROUND(AVG(r.daily_rate_eur), 2) AS avg_daily_rate,
            ROUND(AVG(r.days_held), 1) AS avg_days_held,
            SUM(CASE WHEN r.rental_returned > r.rental_due THEN 1 ELSE 0 END) AS late_returns,
            ROUND(
                100.0 * SUM(CASE WHEN r.rental_returned > r.rental_due THEN 1 ELSE 0 END)
                / NULLIF(COUNT(r.rental_id), 0), 1
            ) AS late_return_pct
        FROM rentals r
        JOIN products p   ON r.product_id  = p.product_id
        JOIN categories c ON p.category_id = c.category_id
        GROUP BY c.name""",
    ]

    with engine.connect() as con:
        for statement in views:
            con.execute(text(statement))
        con.commit()

    # ── CSVs ──────────────────────────────────────────────────
    for table_name, df in tables.items():
        out_path = CSV_DIR / f"{table_name}.csv"

        if not out_path.exists():
            df.to_csv(out_path, index=False)

    # ── Summary ───────────────────────────────────────────────
    print()
    print("=" * 55)
    print("  Data generation complete")
    print("=" * 55)

    summary = pd.read_sql("""
        SELECT category, total_items, avg_days_on_shelf,
               ROUND(stale_inventory_value_eur, 0) AS stale_eur,
               rental_eligible_count
        FROM v_inventory_aging
        ORDER BY stale_eur DESC
    """, engine)
    print("  Inventory aging summary:")
    print(summary.to_string(index=False))

    rev = pd.read_sql("""
        SELECT category, total_rentals,
               ROUND(total_rental_revenue, 0) AS rental_revenue_eur,
               avg_days_held, late_return_pct
        FROM v_revenue_comparison
        ORDER BY rental_revenue_eur DESC
    """, engine)
    print("\n  Revenue by category:")
    print(rev.to_string(index=False))

    headline = pd.read_sql("""
        SELECT
            ROUND(SUM(rental_90d_revenue), 0) AS rental_potential_eur,
            ROUND(SUM(markdown_revenue), 0)   AS markdown_potential_eur,
            ROUND(
                100.0 * (SUM(rental_90d_revenue) - SUM(markdown_revenue))
                / SUM(markdown_revenue), 1
            ) AS rental_upside_pct
        FROM v_rental_eligible
    """, engine)
    print("\n  Core hypothesis (eligible inventory only):")
    print(f"  Rental potential (90d):   €{int(headline['rental_potential_eur'].iloc[0]):,}")
    print(f"  Markdown alternative:     €{int(headline['markdown_potential_eur'].iloc[0]):,}")
    print(f"  Rental upside:            {headline['rental_upside_pct'].iloc[0]}%")
    print("\n  Done. Open MySQL Workbench to explore your data.")

    # -- Integrity checks --
    print("\n  Integrity checks:")
    overlap_check = pd.read_sql("""
        SELECT COUNT(*) AS overlaps FROM (
            SELECT r1.rental_id
            FROM rentals r1
            JOIN rentals r2 ON r1.product_id = r2.product_id
                AND r1.rental_id < r2.rental_id
                AND r1.rental_start < COALESCE(r2.rental_returned, r2.rental_due)
                AND r2.rental_start < COALESCE(r1.rental_returned, r1.rental_due)
        ) t
    """, engine)
    print(f"  Overlapping rentals:      {int(overlap_check['overlaps'].iloc[0])}")

    neg_check = pd.read_sql("SELECT COUNT(*) AS n FROM rentals WHERE days_held < 1", engine)
    print(f"  Negative/zero days_held:  {int(neg_check['n'].iloc[0])}")

    overdue_check = pd.read_sql(
        f"SELECT COUNT(*) AS n FROM rentals WHERE days_held > {MAX_OVERDUE_DAYS} AND rental_returned IS NULL",
        engine
    )
    print(f"  Overdue > {MAX_OVERDUE_DAYS} days:         {int(overdue_check['n'].iloc[0])}")

    print("\n  Done. Open MySQL Workbench to explore your data.")


write_all()

Building tables...
  categories                      8 rows
  products                    1,880 rows
  inventory_events            5,807 rows
  pricing_rules                   8 rows
  customers                     800 rows
  rentals                     3,500 rows
  return_conditions           3,241 rows

  Data generation complete
  Inventory aging summary:
               category  total_items  avg_days_on_shelf  stale_eur  rental_eligible_count
                Laptops          320             1054.6   540575.0                      3
             Projectors          180             1030.8   491945.0                     13
  Cameras & Photography          220             1016.9   405635.0                      8
            Televisions          280             1015.4   374115.0                      4
                 Drones          180              994.2   183675.0                      6
     Smart Home & Audio          260             1030.9   119085.0                      5
        G

NameError: name 'MAX_OVERDUE_DAYS' is not defined

In [ ]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}"
)

df = pd.read_sql("SELECT * FROM v_inventory_aging", engine)

In [ ]:
# See all tables in your database
pd.read_sql("SHOW TABLES", engine)

,Tables_in_rental_final_project
0,categories
1,customers
2,inventory_events
3,pricing_rules
4,products
5,rentals
6,return_conditions
7,v_inventory_aging
8,v_rental_eligible
9,v_rental_history
